# 04 - Preprocessing checks

Goal: check that `scripts/preprocessing.py` does exactly what CLAUDE.md agrees, **before** any model uses it.

- Training rows only; the test rows are never loaded into the analysis.
- Everything learned is fitted on **fold 0 of `fold_r0`**: the training part (folds 1-4) only. Fold 0 plays the validation part.
- This notebook writes nothing to `results/` and logs nothing: it is a check, not an experiment.

What we check:
1. The fixed rules (`"?"` -> `"Unknown"`, native_country -> 3 groups, capital log1p + flag) are correct.
2. The scaler learned its means from the training part only, not from all train rows.
3. No NaNs come out.
4. The output column names.
5. An unseen category becomes an all-zero block.

## 1. Kernel and library versions
`sys.executable` should be `C:\Users\User\venvs\classification\Scripts\python.exe` (kernel "Python (classification)"). If not, stop and switch the kernel.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

print('Python executable:', sys.executable)
print('pandas      ', pd.__version__)
print('numpy       ', np.__version__)
print('scikit-learn', sklearn.__version__)

Python executable: c:\Users\User\venvs\classification\Scripts\python.exe
pandas       3.0.6
numpy        2.4.6
scikit-learn 1.9.1


## 2. Load data, split and folds (read-only); import the preprocessing module
- Same loading and v2 checks as `03_baseline.ipynb`: split row_ids = all positions minus the dropped duplicates (48,790); `cv_folds.csv` row_ids = the train row_ids.
- Only train rows are kept; the full raw table is deleted.
- `scripts/` is added to `sys.path` so `import preprocessing` finds `scripts/preprocessing.py`. The printed path shows which file was imported.

In [2]:
ROOT = Path.cwd()
if not (ROOT / 'data' / 'adult.csv').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'adult.csv'
RESULTS_PATH = ROOT / 'results'
SPLIT_PATH = RESULTS_PATH / 'split_indices.csv'
DROPPED_PATH = RESULTS_PATH / 'dropped_duplicates.csv'
FOLDS_PATH = RESULTS_PATH / 'cv_folds.csv'
SCRIPTS_PATH = ROOT / 'scripts'
for path in [DATA_PATH, SPLIT_PATH, DROPPED_PATH, FOLDS_PATH, SCRIPTS_PATH / 'preprocessing.py']:
    assert path.exists(), f'Not found: {path}'
print('Project root:', ROOT)

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_PATH))
import preprocessing as prep
print('preprocessing module:', prep.__file__)

df = pd.read_csv(DATA_PATH, header=None, names=prep.RAW_COLUMNS, keep_default_na=False)
assert df.shape == (48842, 15), f'Unexpected shape {df.shape}'

split_df = pd.read_csv(SPLIT_PATH)
assert list(split_df.columns) == ['row_id', 'split']
assert split_df['row_id'].is_unique
assert set(split_df['split'].unique()) == {'train', 'test'}

dropped_df = pd.read_csv(DROPPED_PATH)
assert list(dropped_df.columns) == ['row_id', 'kept_row_id']
expected_ids = set(range(len(df))) - set(dropped_df['row_id'])
assert set(split_df['row_id']) == expected_ids, 'Split row_ids are not all positions minus the dropped duplicates'
assert len(split_df) == 48790

train_ids = np.sort(split_df.loc[split_df['split'] == 'train', 'row_id'].to_numpy())
N_TRAIN = len(train_ids)

folds = pd.read_csv(FOLDS_PATH)
assert list(folds.columns) == ['row_id', 'fold_r0', 'fold_r1', 'fold_r2']
assert np.array_equal(folds['row_id'].to_numpy(), train_ids), 'cv_folds.csv row_ids differ from the train row_ids'

train_df = df.iloc[train_ids].reset_index(drop=True)
del df
assert len(train_df) == N_TRAIN
print(f'Train rows: {N_TRAIN:,}')

Project root: c:\Users\User\OneDrive\Desktop\Classification_Model
preprocessing module: c:\Users\User\OneDrive\Desktop\Classification_Model\scripts\preprocessing.py
Train rows: 39,032


## 3. Fold 0 of fold_r0
- `fit_part`: train rows with fold != 0 (the training part of this fold). Everything learned is fitted here.
- `val_part`: train rows with fold == 0 (the validation part). Only transformed, never fitted on.

In [3]:
fold = folds['fold_r0'].to_numpy()
is_val = fold == 0

fit_part = train_df[~is_val]
val_part = train_df[is_val]

print(f'fit_part (folds 1-4): {len(fit_part):,} rows')
print(f'val_part (fold 0):    {len(val_part):,} rows')
assert len(fit_part) + len(val_part) == N_TRAIN

fit_part (folds 1-4): 31,225 rows
val_part (fold 0):    7,807 rows


## 4. The fixed rules
The fixed rules learn nothing from the data, so it is safe to look at them on **all** train rows.
- For each categorical column, the number of `"?"` before must equal the number of `"Unknown"` after (and the raw data must not already contain `"Unknown"`, otherwise the counts would mix).
- `native_country_group` has only the three agreed values, and `United-States` really occurs (a spelling mismatch would silently send everyone to `Other`).
- Flags are exactly 0/1 and equal 1 exactly when the raw value is > 0. The log1p columns are never negative.
- No NaNs.

In [4]:
fixed_all = prep.apply_fixed_rules(train_df)
print('Fixed-rules output shape:', fixed_all.shape)
assert list(fixed_all.columns) == prep.FEATURE_COLS
print()

print('"?" before vs "Unknown" after:')
for raw_col in prep.RAW_CATEGORICAL_COLS + ['native_country']:
    out_col = 'native_country_group' if raw_col == 'native_country' else raw_col
    n_question = int((train_df[raw_col] == prep.MISSING).sum())
    n_unknown_raw = int((train_df[raw_col] == prep.UNKNOWN).sum())
    n_unknown_out = int((fixed_all[out_col] == prep.UNKNOWN).sum())
    print(f'  {raw_col:<15} "?": {n_question:>5}   "Unknown" after: {n_unknown_out:>5}')
    assert n_unknown_raw == 0, f'{raw_col} already contains "Unknown" in the raw data'
    assert n_question == n_unknown_out, f'{raw_col}: counts differ'
print()

print('native_country_group:')
print(fixed_all['native_country_group'].value_counts().to_string())
assert set(fixed_all['native_country_group'].unique()) <= {prep.US, prep.OTHER, prep.UNKNOWN}
assert (fixed_all['native_country_group'] == prep.US).sum() > 0, 'No United-States rows: check the spelling'
print()

for raw_col in ['capital_gain', 'capital_loss']:
    flag = fixed_all[f'{raw_col}_pos']
    assert set(flag.unique()) <= {0, 1}, f'{raw_col}_pos is not 0/1'
    assert (flag.to_numpy() == (train_df[raw_col].to_numpy() > 0)).all(), f'{raw_col}_pos does not match value > 0'
    assert (fixed_all[f'{raw_col}_log1p'] >= 0).all(), f'{raw_col}_log1p has negative values'
    print(f'{raw_col}_pos share of 1s: {flag.mean():.2%}')

assert not fixed_all.isna().any().any(), 'Fixed-rules output has NaNs'
print()
print('Fixed rules: all checks passed.')

Fixed-rules output shape: (39032, 14)

"?" before vs "Unknown" after:
  workclass       "?":  2255   "Unknown" after:  2255
  marital_status  "?":     0   "Unknown" after:     0
  occupation      "?":  2263   "Unknown" after:  2263
  relationship    "?":     0   "Unknown" after:     0
  race            "?":     0   "Unknown" after:     0
  sex             "?":     0   "Unknown" after:     0
  native_country  "?":   708   "Unknown" after:   708

native_country_group:
native_country_group
United-States    34979
Other             3345
Unknown            708

capital_gain_pos share of 1s: 8.30%
capital_loss_pos share of 1s: 4.70%

Fixed rules: all checks passed.


## 5. Fit on the training part only; where did the scaler learn its means?
We fit a fresh preprocessor on `fit_part` only, then compare the scaler's learned means with:
- the means of the same columns on `fit_part`: they **must match** (that is what it learned from);
- the means on **all** train rows (which include the validation fold): they **must not** match exactly.

The differences will be tiny, because the folds are random and stratified. That is expected: the point is *which rows* the scaler learned from, not how big the difference is. If the scaler had seen the validation rows, they would match exactly.

After scaling, the numeric columns should have mean ~0 and std ~1 on `fit_part` (StandardScaler uses the population std, `ddof=0`), but only approximately on `val_part`, because `val_part` was not used to learn the scaling.

In [5]:
pre = prep.build_preprocessor()
pre.fit(fit_part)

scaler = pre.named_steps['learned'].named_transformers_['num'].named_steps['scale']
fixed_fit = prep.apply_fixed_rules(fit_part)

mean_table = pd.DataFrame({
    'scaler_mean': scaler.mean_,
    'fit_part_mean': fixed_fit[prep.NUMERIC_COLS].mean().to_numpy(),
    'all_train_mean': fixed_all[prep.NUMERIC_COLS].mean().to_numpy(),
}, index=prep.NUMERIC_COLS)
mean_table['scaler_minus_all_train'] = mean_table['scaler_mean'] - mean_table['all_train_mean']
print(mean_table.round(6).to_string())

assert np.allclose(mean_table['scaler_mean'], mean_table['fit_part_mean']), 'Scaler means differ from the fit_part means'
assert (mean_table['scaler_minus_all_train'] != 0).any(), 'Scaler means equal the all-train means exactly: did it see the validation fold?'
print()
print('Scaler learned from fit_part only.')
print()

feature_names = pre.get_feature_names_out()
Z_fit = pre.transform(fit_part)
Z_val = pre.transform(val_part)
num_idx = [list(feature_names).index(f'num__{c}') for c in prep.NUMERIC_COLS]

scaled_stats = pd.DataFrame({
    'fit_mean': Z_fit[:, num_idx].mean(axis=0),
    'fit_std': Z_fit[:, num_idx].std(axis=0),
    'val_mean': Z_val[:, num_idx].mean(axis=0),
    'val_std': Z_val[:, num_idx].std(axis=0),
}, index=prep.NUMERIC_COLS)
print('Scaled numeric columns:')
print(scaled_stats.round(4).to_string())

                    scaler_mean  fit_part_mean  all_train_mean  scaler_minus_all_train
age                   38.624307      38.624307       38.576424                0.047883
education_num         10.078399      10.078399       10.086519               -0.008120
hours_per_week        40.377742      40.377742       40.417196               -0.039454
capital_gain_log1p     0.737977       0.737977        0.730753                0.007224
capital_loss_log1p     0.356490       0.356490        0.352771                0.003719

Scaler learned from fit_part only.

Scaled numeric columns:
                    fit_mean  fit_std  val_mean  val_std
age                     -0.0      1.0   -0.0175   0.9844
education_num            0.0      1.0    0.0158   0.9906
hours_per_week          -0.0      1.0    0.0159   0.9835
capital_gain_log1p      -0.0      1.0   -0.0147   0.9777
capital_loss_log1p      -0.0      1.0   -0.0116   0.9751


## 6. No NaNs, right shape, flags still 0/1
Both parts are transformed by the preprocessor fitted on `fit_part`. The output must have one row per input row, one column per output name, no NaNs, and the flag columns must still be exactly 0/1 (they are passed through unscaled).

In [6]:
flag_idx = [list(feature_names).index(f'flag__{c}') for c in prep.FLAG_COLS]

for name, Z, part in [('fit_part', Z_fit, fit_part), ('val_part', Z_val, val_part)]:
    assert Z.shape == (len(part), len(feature_names)), f'{name}: unexpected shape {Z.shape}'
    n_nan = int(np.isnan(Z).sum())
    assert n_nan == 0, f'{name}: {n_nan} NaNs in the output'
    assert set(np.unique(Z[:, flag_idx])) <= {0.0, 1.0}, f'{name}: flags are not 0/1'
    print(f'{name}: shape {Z.shape}, NaNs: {n_nan}, flags 0/1: yes')

fit_part: shape (31225, 54), NaNs: 0, flags 0/1: yes
val_part: shape (7807, 54), NaNs: 0, flags 0/1: yes


## 7. Output column names
`get_feature_names_out()` gives one name per output column, prefixed by the branch: `num__` (scaled), `flag__` (passed through), `cat__<column>_<category>` (one-hot). The one-hot columns are the categories seen in `fit_part`.

In [7]:
names = list(feature_names)
print('Numeric (scaled):', [n for n in names if n.startswith('num__')])
print('Flags (unscaled):', [n for n in names if n.startswith('flag__')])
print()
n_cat_total = 0
for c in prep.CATEGORICAL_COLS:
    block = [n for n in names if n.startswith(f'cat__{c}_')]
    n_cat_total += len(block)
    print(f'{c} ({len(block)}):')
    print('   ', block)
print()
assert len(prep.NUMERIC_COLS) + len(prep.FLAG_COLS) + n_cat_total == len(names), 'Some output columns are not accounted for'
print('Total output columns:', len(names))

Numeric (scaled): ['num__age', 'num__education_num', 'num__hours_per_week', 'num__capital_gain_log1p', 'num__capital_loss_log1p']
Flags (unscaled): ['flag__capital_gain_pos', 'flag__capital_loss_pos']

workclass (9):
    ['cat__workclass_Federal-gov', 'cat__workclass_Local-gov', 'cat__workclass_Never-worked', 'cat__workclass_Private', 'cat__workclass_Self-emp-inc', 'cat__workclass_Self-emp-not-inc', 'cat__workclass_State-gov', 'cat__workclass_Unknown', 'cat__workclass_Without-pay']
marital_status (7):
    ['cat__marital_status_Divorced', 'cat__marital_status_Married-AF-spouse', 'cat__marital_status_Married-civ-spouse', 'cat__marital_status_Married-spouse-absent', 'cat__marital_status_Never-married', 'cat__marital_status_Separated', 'cat__marital_status_Widowed']
occupation (15):
    ['cat__occupation_Adm-clerical', 'cat__occupation_Armed-Forces', 'cat__occupation_Craft-repair', 'cat__occupation_Exec-managerial', 'cat__occupation_Farming-fishing', 'cat__occupation_Handlers-cleaners', 'c

## 8. An unseen category gives an all-zero block
We copy one `val_part` row and set `workclass` to a value never seen in training. With `handle_unknown="ignore"`, the whole `workclass` one-hot block must become 0, and every other output column must be unchanged. scikit-learn may print a warning about the unknown category; that is expected here.

Contrast: an unknown **country** (e.g. `"Atlantis"`) is not an unseen category for the encoder, because the fixed mapping turns it into `Other` before the encoder sees it.

In [8]:
row = val_part.iloc[[0]].copy()
z_orig = pre.transform(row)

row_unseen = row.copy()
row_unseen['workclass'] = 'Never-seen-category'
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    z_unseen = pre.transform(row_unseen)
for w in caught:
    print('Warning (expected):', w.message)

wc_idx = [i for i, n in enumerate(names) if n.startswith('cat__workclass_')]
other_idx = [i for i in range(len(names)) if i not in wc_idx]

print('Original workclass:', row['workclass'].iloc[0])
print('workclass block before:', z_orig[0, wc_idx])
print('workclass block after: ', z_unseen[0, wc_idx])
assert z_orig[0, wc_idx].sum() == 1, 'Original row should have exactly one active workclass column'
assert (z_unseen[0, wc_idx] == 0).all(), 'Unseen category did not give an all-zero block'
assert np.array_equal(z_unseen[0, other_idx], z_orig[0, other_idx]), 'Other columns changed'
print('Unseen workclass -> all-zero block; all other columns unchanged.')
print()

row_atlantis = row.copy()
row_atlantis['native_country'] = 'Atlantis'
z_atlantis = pre.transform(row_atlantis)
nc_idx = [i for i, n in enumerate(names) if n.startswith('cat__native_country_group_')]
active = [names[i] for i in nc_idx if z_atlantis[0, i] == 1]
print('native_country = "Atlantis" -> active column:', active)
assert active == ['cat__native_country_group_Other'], 'Atlantis should map to Other'

Original workclass: Private
workclass block before: [0. 0. 0. 1. 0. 0. 0. 0. 0.]
workclass block after:  [0. 0. 0. 0. 0. 0. 0. 0. 0.]
Unseen workclass -> all-zero block; all other columns unchanged.

native_country = "Atlantis" -> active column: ['cat__native_country_group_Other']
